In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Optional: For better display
pd.set_option('display.max_rows', 20)

In [ ]:
# Load the training base (the one that already has themes)
# We drop rows where 'themes' is empty because we can't learn from them
df_train = pd.read_excel('base_with_themes.xlsx')
df_train = df_train.dropna(subset=['themes'])

# Load the target base (the one with gaps to fill)
df_target = pd.read_excel('base_to_fill.xlsx')

print(f"Training on {len(df_train)} labeled examples.")
print(f"Predicting for {len(df_target)} entries.")

In [ ]:
# Define the pipeline
pipeline = Pipeline([
    # Step 1: Turn text codes into numbers (vectors) based on character patterns
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5))),
    
    # Step 2: The Classifier (Random Forest is robust for this type of data)
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Define features (X) and target (y)
X_train = df_train['xcats'].astype(str) # Ensure it's treated as text
y_train = df_train['themes']

# Fit (Train) the model
print("Training the model... this may take a moment.")
pipeline.fit(X_train, y_train)
print("Training complete!")

In [ ]:
# Split the training data just to test accuracy
X_t, X_v, y_t, y_v = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Train on the subset
pipeline.fit(X_t, y_t)

# Score
accuracy = pipeline.score(X_v, y_v)
print(f"Model Accuracy on internal test: {accuracy:.1%}")

In [ ]:
# Retrain on ALL available data for maximum accuracy
pipeline.fit(X_train, y_train)

# Identify rows in the target file where 'themes' is missing (NaN)
# If you want to overwrite ALL themes (even existing ones), skip the line below
mask_missing = df_target['themes'].isna() 

# Predict only for the missing rows
# We ensure the input is string format
X_missing = df_target.loc[mask_missing, 'xcats'].astype(str)

if len(X_missing) > 0:
    predicted_themes = pipeline.predict(X_missing)
    
    # Fill the gaps
    df_target.loc[mask_missing, 'themes'] = predicted_themes
    print(f"Successfully filled {len(X_missing)} rows.")
else:
    print("No missing themes found to fill.")

# Show a sample of the filled data
df_target[mask_missing].head()

In [ ]:
df_target.to_excel('filled_database.xlsx', index=False)
print("File saved as 'filled_database.xlsx'")